# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsf-rawnak/FlyRankAI-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

FlyRank's clients don't have a shortage of pages that need attention — they have a shortage of
reviewer hours. A content team can look at maybe 20-50 pages a week, not 13,000. The job isn't
"find problems" (the data below shows problems are everywhere), it's "rank the problems so the
first 50 a human looks at are the ones most worth their time." That's a ranking/scoring problem,
which is exactly what this lane is built for, and it's the lane with the clearest path from
"page" to "action someone actually takes."

It's also the lowest-risk lane for a 7-week build: the starter pipeline in this repo already
proves the approach works end to end on this exact dataset (baseline rules score 0.627 ROC-AUC
and get 24% of their top-50 picks right; a random forest gets to 0.750 ROC-AUC and 74% of its
top-50 right). I'm not betting 7 weeks on an unproven idea — I'm starting from a working
baseline and trying to earn a real improvement on top of it, with my own future-window label
instead of the current-window proxy the starter uses.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Given a content item's prior 90-day search and engagement signals, which
pages should a content reviewer look at first this week, and what should they do with each one
(refresh, expand, protect, prune, or monitor)?

**Unit of analysis:** one content item (`content_id`) at one point in time — the grain matches
the starter dataset: one row per pseudonymized page, trailing-90-day metrics.

**Decision this improves:** which pages a content reviewer opens first out of a backlog that is
always bigger than their available hours.

**Who acts on it:** a content/SEO reviewer (or their manager, deciding where to assign reviewer
time). They open the ranked queue, read the reason codes, and either refresh the page's content,
expand it, protect it from a change, deprioritize/prune it, or just keep monitoring it.

**Output:** a ranked review queue — score, action suggestion, and reason code per page.

**Cost of a wrong call:**
- *False positive* (page ranked high, isn't actually worth fixing): wasted reviewer hours — the
  team spends a session on a page that wouldn't have moved the needle, and a genuinely declining
  page waits another week.
- *False negative* (a real decline sits low in the queue, never gets reviewed): the page keeps
  losing visibility uninterrupted, and by the time anyone notices it's a bigger recovery job than
  if it had been caught early.
- Neither error is catastrophic on its own (nobody loses money in one bad ranking), but reviewer
  time is the scarcest resource in this system, so the cost is measured in wasted or misallocated
  hours, not in a single dramatic failure.

**Why data/ML helps instead of a plain rule:** a single if-statement rule already exists (the
starter baseline) and it's genuinely useful — but it treats every signal with a fixed weight
(0.40 visibility, 0.30 freshness, 0.25 position, 0.05 depth) regardless of how those signals
actually interact for a given content type or client. The signals here are tangled — a page can
be old AND high-traffic AND well-positioned all at once, and which combination actually predicts
future decline isn't obvious by eye. That's the kind of messy-but-real pattern where a learned
model has room to beat a fixed rule, which the starter results already show (0.240 to 0.740
precision@50 baseline vs. random forest).


## 3. Quick look at the data (2-3 real numbers)

Loaded the starter dataset below. Three numbers that tell me this lane is worth the next 7 weeks:

1. **13,152 of 30,000 pages (43.8%)** already match a `declining_with_demand` pattern
   (`trend_direction == "down"` and `impressions_90d >= 100`) — a real backlog, not a toy problem.
2. Across **32 clients**, no reviewer team is opening 13,000+ pages a week — which is exactly why
   ranking, not just flagging, is the actual job here.
3. The starter pipeline (already run and committed in `outputs/model_report.md`) shows a learned
   model roughly **triples** the baseline rule's precision at the top of the queue: 24% of the
   baseline's top 50 picks are correct vs. 74% for the random forest. That's the gap I'm trying
   to earn on a stronger, future-looking label instead of the current-window proxy label the
   starter uses.


In [1]:
import pandas as pd
from pathlib import Path

local_path = Path("../../data/raw/content_refresh_anonymized.csv")
raw_url = "https://raw.githubusercontent.com/rsf-rawnak/FlyRankAI-ML-Internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(local_path) if local_path.exists() else pd.read_csv(raw_url)

n_total = len(df)

declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
thin_visible = df[(df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)]

print(f"Total pages in starter dataset: {n_total:,}")
print(f"Clients: {df['client_id'].nunique()}")
print()
print(f"declining_with_demand: {len(declining_with_demand):,} pages ({100*len(declining_with_demand)/n_total:.1f}%)")
print(f"stale_visible_page:    {len(stale_visible):,} pages ({100*len(stale_visible)/n_total:.1f}%)")
print(f"thin_visible_page:     {len(thin_visible):,} pages ({100*len(thin_visible)/n_total:.1f}%)")


Total pages in starter dataset: 30,000
Clients: 32

declining_with_demand: 13,152 pages (43.8%)
stale_visible_page:    17 pages (0.1%)
thin_visible_page:     82 pages (0.3%)


## 4. Careful words: what I can and can't claim

**What I can claim:**
- Observed patterns in trailing-90-day search and engagement metrics for this anonymized slice
  of 30,000 pages across 32 clients.
- A ranked queue that is decision-support: it orders review candidates by evidence, it does not
  guarantee any individual page's outcome.
- Directional, measured comparisons against a transparent baseline rule (precision@K, ROC-AUC,
  average precision) on a proper holdout — ideally client-held-out and, if I move to the
  warehouse release, time-aware as well.
- That a learned ranking outperforms — or fails to outperform — the fixed-weight baseline rule,
  by a stated metric, on stated data.

**What I will never claim:**
- That refreshing a page *caused* a recovery — that needs a controlled experiment, not this
  observational data.
- Anything about Google's actual ranking algorithm or "AI ranking/citation" behavior — this data
  only shows search and session outcomes, never the mechanism behind them.
- That the model's label is ground truth — `is_declining_label`/`trend_direction` is a
  current-window proxy, not an observed future outcome, so I plan to move toward a genuine
  prior-window → future-window label as the project matures, and I'll say clearly which one I'm
  using at each stage.
- That a high score means a page is guaranteed worth fixing — it means a page is worth a human
  reviewer's look, given limited capacity.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.